In [7]:
import requests
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
import xml.etree.ElementTree as ET
import re
import random
import zipfile

In [8]:
# =========================================================
# PATHS
# =========================================================

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

import sys
sys.path.insert(0, str(BASE_DIR / "src"))

RAW_DIR = BASE_DIR / "data" / "raw"
CACHE_DIR = BASE_DIR / "data" / "caches"
BUILD_DIR = BASE_DIR / "data" / "build"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
LOG_DIR = BASE_DIR / "logs"

for directory in [RAW_DIR, CACHE_DIR, BUILD_DIR, PROCESSED_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MAL_IDS_CACHE_URL = (
    "https://raw.githubusercontent.com/purarue/mal-id-cache/"
    "master/cache/anime_cache.json"
)
ANIDB_XML_CACHE_URL = (
    "https://files.shokoanime.com/files/shoko-server/other/Anime_HTTP.zip"
)

MAL_IDS_CACHE_FILE = RAW_DIR / "mal_candidate_ids.json"
LEGACY_MAL_IDS_CACHE_FILE = RAW_DIR / "anime_cache.json"

ANIDB_CACHE_FILE = CACHE_DIR / "anidb_metadata_cache.json"
LEGACY_ANIDB_CACHE_FILE = RAW_DIR / "anidb_cache.json"
ANIDB_XML_CACHE_ZIP = RAW_DIR / "Anime_HTTP.zip"

OUTPUT_CSV = PROCESSED_DIR / "anime_dataset.csv"
OUTPUT_JSON = PROCESSED_DIR / "anime_dataset.json"

CHECKPOINT_FILE = BUILD_DIR / "dataset_checkpoint.json"
FAILED_IDS_FILE = BUILD_DIR / "failed_api_requests.json"
FILTERED_INVALID_TYPES_FILE = BUILD_DIR / "skipped_invalid_type_ids.json"
PERMANENT_HTTP_SKIP_FILE = BUILD_DIR / "skipped_permanent_http_ids.json"
BUILD_SUMMARY_FILE = BUILD_DIR / "dataset_build_summary.json"

LEGACY_CHECKPOINT_FILE = LOG_DIR / "dataset_checkpoint.json"
LEGACY_FAILED_IDS_FILE = LOG_DIR / "failed_anime_ids.json"
LEGACY_FILTERED_INVALID_TYPES_FILE = LOG_DIR / "filtered_invalid_type_ids.json"
LEGACY_PERMANENT_HTTP_SKIP_FILE = LOG_DIR / "permanent_http_skip_ids.json"
LEGACY_BUILD_SUMMARY_FILE = LOG_DIR / "dataset_build_summary.json"

LOG_FILE = LOG_DIR / "dataset_build_log.txt"


def migrate_file_if_needed(old_path, new_path):
    old_path = Path(old_path)
    new_path = Path(new_path)

    if new_path.exists() or not old_path.exists():
        return

    new_path.parent.mkdir(parents=True, exist_ok=True)
    old_path.replace(new_path)
    print(f"Migrated {old_path} -> {new_path}")


for old_path, new_path in [
    (LEGACY_MAL_IDS_CACHE_FILE, MAL_IDS_CACHE_FILE),
    (LEGACY_ANIDB_CACHE_FILE, ANIDB_CACHE_FILE),
    (LEGACY_CHECKPOINT_FILE, CHECKPOINT_FILE),
    (LEGACY_FAILED_IDS_FILE, FAILED_IDS_FILE),
    (LEGACY_FILTERED_INVALID_TYPES_FILE, FILTERED_INVALID_TYPES_FILE),
    (LEGACY_PERMANENT_HTTP_SKIP_FILE, PERMANENT_HTTP_SKIP_FILE),
    (LEGACY_BUILD_SUMMARY_FILE, BUILD_SUMMARY_FILE),
]:
    migrate_file_if_needed(old_path, new_path)


In [9]:
# =========================================================
# LOAD MAL IDS
# =========================================================

def download_mal_candidate_ids():
    tmp_path = MAL_IDS_CACHE_FILE.with_suffix(".json.tmp")

    try:
        response = requests.get(
            MAL_IDS_CACHE_URL,
            timeout=60,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        response.raise_for_status()

        payload = response.json()

        if "sfw" not in payload or "nsfw" not in payload:
            raise ValueError("Downloaded MAL ID cache missing sfw/nsfw keys")

        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        tmp_path.replace(MAL_IDS_CACHE_FILE)
        print(f"Downloaded MAL candidate IDs: {MAL_IDS_CACHE_FILE}")

    except Exception as exc:
        if tmp_path.exists():
            tmp_path.unlink()

        if MAL_IDS_CACHE_FILE.exists():
            print(f"MAL ID download failed, using existing cache: {exc}")
        else:
            raise


download_mal_candidate_ids()

with open(MAL_IDS_CACHE_FILE, "r", encoding="utf-8") as f:
    cache = json.load(f)

sfw_ids = cache["sfw"]
nsfw_ids = cache["nsfw"]

print(f"SFW IDs: {len(sfw_ids)}")
print(f"NSFW IDs: {len(nsfw_ids)}")


Downloaded MAL candidate IDs: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\mal_candidate_ids.json
SFW IDs: 28390
NSFW IDs: 1623


In [10]:
# =========================================================
# FULL DATASET / SMALL TEST SAMPLE
# =========================================================

TEST_MODE = False
TEST_SFW_LIMIT = 20
TEST_NSFW_LIMIT = 20

if TEST_MODE:
    sfw_ids = sfw_ids[:TEST_SFW_LIMIT]
    nsfw_ids = nsfw_ids[:TEST_NSFW_LIMIT]

all_entries = (
    [(mid, False) for mid in sfw_ids]
    + [(mid, True) for mid in nsfw_ids]
)
print(f"SFW entries selected: {len(sfw_ids)}")
print(f"NSFW entries selected: {len(nsfw_ids)}")
print(f"Total entries to process: {len(all_entries)}")

SFW entries selected: 28390
NSFW entries selected: 1623
Total entries to process: 30013


In [11]:
# =========================================================
# FILTER RULES
# =========================================================

VALID_TYPES = {"TV", "Movie", "OVA", "ONA", "Special", "TV Special"}

VALID_STATUS = {
    "Finished Airing",
    "Currently Airing"
}

ANIDB_CLIENT = "matuki"
ANIDB_CLIENTVER = 3
ANIDB_REQUEST_DELAY_SECONDS = 4
ANIDB_REQUEST_JITTER_SECONDS = 0.5
ANIDB_COOLDOWN_SECONDS = 30 * 60
ANIDB_LIVE_CACHE_TTL_DAYS = 30


In [12]:
# =========================================================
# HELPER FUNCTIONS
# =========================================================

from anidb_metadata_utils import extract_anidb_payload

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504, 520, 522, 524}
PERMANENT_HTTP_SKIP_STATUS_CODES = {404, 410}

build_stats = {
    "processed": 0,
    "added": 0,
    "filtered": 0,
    "failed": 0,
    "skipped_existing": 0,
    "skipped_invalid_type": 0,
    "skipped_permanent_http": 0,
    "anidb_episode_fills": 0,
    "anidb_tag_fills": 0,
    "anidb_explicit_fills": 0,
    "anidb_demographic_fills": 0,
    "anidb_cache_hits": 0,
    "anidb_live_requests": 0,
    "anidb_cooldowns": 0,
    "anidb_cooldown_skips": 0,
    "anidb_ban_resets": 0,
    "shoko_xml_cache_imported": 0,
    "shoko_xml_cache_skipped_existing": 0,
    "shoko_xml_cache_parse_errors": 0,
    "recommendation_fills": 0,
}

ANIDB_LIVE_REQUESTS_IN_WINDOW = 0
ANIDB_COOLDOWN_UNTIL_TS = 0

IGNORED_ANIDB_BRANCHES = {
    "maintenance tags",
    "origin",
    "original work",
    "technical aspects",
    "setting",
}

DROPPED_ANIDB_TAGS = {
    "warning",
}

DEMOGRAPHIC_BRANCH = "target audience"

ALWAYS_EXPLICIT_BRANCHES = {
    "pornography",
    "sexual abuse",
    "rape",
}

RATING_GATED_EXPLICIT_BRANCHES = {
    "fetishes",
    "ecchi",
    "incest",
    "brainwashing",
    "harem",
    "content indicators",
}

RATING_GATED_EXPLICIT_TAGS = {
    "nudity",
    "sex",
    "animal abuse",
    "gore",
    "mutilation",
    "ecchi",
    "incest",
    "brainwashing",
    "harem",
}


def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def write_log(message):
    timestamp = datetime.now()
    line = f"[{timestamp}] {message}"
    print(line)

    with open(LOG_FILE, "a", encoding="utf-8") as log:
        log.write(line + "\n")


def write_section_log(title, details=None):
    write_log("=" * 72)
    write_log(title)

    if details:
        for key, value in details.items():
            write_log(f"{key}: {value}")

    write_log("=" * 72)


def json_safe(value):
    if isinstance(value, dict):
        return {
            key: json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [json_safe(item) for item in value]

    if pd.isna(value):
        return None

    return value


def atomic_write_json(path, payload):
    path = Path(path)
    tmp_path = path.with_name(
        f"{path.name}.{int(time.time() * 1000)}.{random.randint(1000, 9999)}.tmp"
    )

    compact_cache_json = path.name == ANIDB_CACHE_FILE.name

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(
            json_safe(payload),
            f,
            ensure_ascii=False,
            indent=None if compact_cache_json else 2,
            separators=(",", ":") if compact_cache_json else None,
            allow_nan=False,
        )

    tmp_path.replace(path)


def atomic_write_csv(df, path):
    path = Path(path)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def failure_key(mal_id, stage):
    return f"{int(mal_id)}::{stage}"


def load_failed_registry():
    if not FAILED_IDS_FILE.exists():
        return {}

    with open(FAILED_IDS_FILE, "r", encoding="utf-8") as f:
        payload = json.load(f)

    failures = payload.get("failures", [])
    registry = {}

    for item in failures:
        if item.get("mal_id") is None:
            continue

        stage = item.get("stage", "unknown")
        registry[failure_key(item["mal_id"], stage)] = item

    return registry


def save_failed_registry():
    payload = {
        "updated_at": now_iso(),
        "failures": sorted(
            failed_registry.values(),
            key=lambda item: (int(item["mal_id"]), item.get("stage", ""))
        )
    }

    atomic_write_json(FAILED_IDS_FILE, payload)


def compact_invalid_type_item(item):
    return {
        "mal_id": int(item["mal_id"]),
        "anime_type": item.get("anime_type"),
        "index": item.get("index"),
    }


def load_invalid_type_registry():
    if not FILTERED_INVALID_TYPES_FILE.exists():
        return {}

    with open(FILTERED_INVALID_TYPES_FILE, "r", encoding="utf-8") as f:
        payload = json.load(f)

    return {
        int(item["mal_id"]): compact_invalid_type_item(item)
        for item in payload.get("invalid_type_ids", [])
        if item.get("mal_id") is not None
    }


def save_invalid_type_registry():
    payload = {
        "invalid_type_ids": [
            compact_invalid_type_item(invalid_type_registry[mal_id])
            for mal_id in sorted(invalid_type_registry)
        ]
    }

    atomic_write_json(FILTERED_INVALID_TYPES_FILE, payload)


def record_invalid_type_id(mal_id, is_nsfw, anime_type, title=None, idx=None):
    mal_id = int(mal_id)

    invalid_type_registry[mal_id] = {
        "mal_id": mal_id,
        "anime_type": anime_type,
        "index": idx,
    }

    save_invalid_type_registry()
    write_log(
        f"FILTERED_INVALID_TYPE_SAVED | MAL {mal_id} | type={anime_type}"
    )


def is_known_invalid_type(mal_id):
    return int(mal_id) in invalid_type_registry


def compact_permanent_http_skip_item(item):
    compact = {
        "mal_id": int(item["mal_id"]),
        "status_code": item.get("status_code"),
        "index": item.get("index"),
    }

    for key in ["reason", "anidb_id"]:
        if item.get(key) is not None:
            compact[key] = item.get(key)

    return compact


def load_permanent_http_skip_registry():
    if not PERMANENT_HTTP_SKIP_FILE.exists():
        return {}

    with open(PERMANENT_HTTP_SKIP_FILE, "r", encoding="utf-8") as f:
        payload = json.load(f)

    return {
        int(item["mal_id"]): compact_permanent_http_skip_item(item)
        for item in payload.get("permanent_http_skip_ids", [])
        if item.get("mal_id") is not None
    }


def save_permanent_http_skip_registry():
    payload = {
        "permanent_http_skip_ids": [
            compact_permanent_http_skip_item(permanent_http_skip_registry[mal_id])
            for mal_id in sorted(permanent_http_skip_registry)
        ]
    }

    atomic_write_json(PERMANENT_HTTP_SKIP_FILE, payload)


def record_permanent_http_skip_id(
    mal_id,
    status_code,
    idx=None,
    reason=None,
    anidb_id=None,
):
    mal_id = int(mal_id)
    item = {
        "mal_id": mal_id,
        "status_code": status_code,
        "index": idx,
    }

    if reason:
        item["reason"] = reason

    if anidb_id:
        item["anidb_id"] = int(anidb_id)

    permanent_http_skip_registry[mal_id] = item
    save_permanent_http_skip_registry()

    if reason:
        write_log(
            f"PERMANENT_SKIP_SAVED | MAL {mal_id} | "
            f"reason={reason} | anidb_id={anidb_id}"
        )
    else:
        write_log(
            f"PERMANENT_HTTP_SKIP_SAVED | MAL {mal_id} | HTTP {status_code}"
        )


def is_known_permanent_http_skip(mal_id):
    item = permanent_http_skip_registry.get(int(mal_id))
    if not item:
        return False

    return item.get("status_code") in PERMANENT_HTTP_SKIP_STATUS_CODES


def is_known_permanent_anidb_skip(mal_id, anidb_id=None):
    item = permanent_http_skip_registry.get(int(mal_id))
    if not item:
        return False

    if item.get("reason") != "anidb_anime_not_found":
        return False

    if anidb_id and item.get("anidb_id"):
        return int(item["anidb_id"]) == int(anidb_id)

    return True


def is_empty_anidb_metadata(metadata):
    normalized = normalize_anidb_metadata(metadata)
    return (
        normalized["episode_count"] is None
        and not normalized["tags"]
        and not normalized["tag_weights"]
        and not normalized["explicit_tags"]
        and not normalized["explicit_tag_weights"]
        and not normalized["demographics"]
    )


def is_anidb_anime_not_found_response(root):
    if root.tag.lower() != "error":
        return False

    text = (root.text or "").strip().casefold()
    return "anime not found" in text or "unknown anime id" in text


def relation_mal_ids_from_text(relations):
    if pd.isna(relations) or not str(relations).strip():
        return []

    related_ids = []
    for item in str(relations).split("|"):
        _, _, related_id = item.partition(":")
        try:
            related_ids.append(int(related_id))
        except ValueError:
            continue

    return related_ids


def select_related_anidb_id_from_dataset(
    mal_id,
    relations,
    excluded_anidb_id=None,
):
    if not OUTPUT_CSV.exists():
        return None

    try:
        df = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))
    except Exception as e:
        write_log(
            f"ANIDB_RELATED_FALLBACK_SKIPPED | MAL {mal_id} | "
            f"could not read dataset: {e}"
        )
        return None

    if "mal_id" not in df.columns or "anidb_id" not in df.columns:
        return None

    related_mal_ids = set(relation_mal_ids_from_text(relations))

    if "relations" in df.columns:
        reverse_links = df[
            df["relations"].fillna("").apply(
                lambda value: int(mal_id) in relation_mal_ids_from_text(value)
            )
        ]
        related_mal_ids.update(reverse_links["mal_id"].astype(int).tolist())

    if not related_mal_ids:
        return None

    candidates = df[df["mal_id"].astype(int).isin(related_mal_ids)].copy()
    candidates = candidates[candidates["anidb_id"].notna()]

    if excluded_anidb_id:
        candidates = candidates[
            candidates["anidb_id"].astype(int) != int(excluded_anidb_id)
        ]

    if candidates.empty:
        return None

    sort_columns = []
    ascending = []

    if "members" in candidates.columns:
        sort_columns.append("members")
        ascending.append(False)

    if "score" in candidates.columns:
        sort_columns.append("score")
        ascending.append(False)

    if "popularity" in candidates.columns:
        sort_columns.append("popularity")
        ascending.append(True)

    if sort_columns:
        candidates = candidates.sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last",
        )

    selected = candidates.iloc[0]
    fallback_anidb_id = int(selected["anidb_id"])
    write_log(
        f"ANIDB_RELATED_FALLBACK_SELECTED | MAL {mal_id} | "
        f"AniDB {excluded_anidb_id} -> {fallback_anidb_id} | "
        f"related_mal={int(selected['mal_id'])} | title={selected.get('title')}"
    )
    return fallback_anidb_id


def empty_anidb_metadata():
    return {
        "episode_count": None,
        "tags": "",
        "tag_weights": "",
        "explicit_tags": "",
        "explicit_tag_weights": "",
        "demographics": "",
    }


def normalize_anidb_metadata(metadata):
    normalized = empty_anidb_metadata()

    if metadata:
        normalized.update({
            key: metadata.get(key, value)
            for key, value in normalized.items()
        })

    return normalized


def load_anidb_cache():
    if not ANIDB_CACHE_FILE.exists():
        return {}

    with open(ANIDB_CACHE_FILE, "r", encoding="utf-8") as f:
        payload = json.load(f)

    return payload.get("items", {})


def save_anidb_cache():
    payload = {
        "updated_at": now_iso(),
        "items": anidb_cache,
    }

    atomic_write_json(ANIDB_CACHE_FILE, payload)


def cache_anidb_payload(anidb_id, payload, source="live_http", save=True):
    if not anidb_id:
        return

    # For Shoko's XML cache, freshness should reflect the XML file timestamp,
    # not the day we downloaded/imported the zip.
    cached_at = (
        payload.get("xml_file_modified_at")
        if source == "shoko_xml_cache" and payload.get("xml_file_modified_at")
        else now_iso()
    )

    cache_entry = {
        "cached_at": cached_at,
        "source": source,
        "episode_count": payload.get("episode_count"),
        "raw_tags": payload.get("raw_tags", []),
    }

    for key in [
        "xml_file_modified_at",
        "source_url",
        "source_downloaded_at",
    ]:
        if payload.get(key):
            cache_entry[key] = payload[key]

    anidb_cache[str(anidb_id)] = cache_entry
    if save:
        save_anidb_cache()


def cache_anidb_metadata(anidb_id, metadata):
    """
    Backward-compatible writer for older cache entries. New entries should use
    cache_anidb_payload so explicit classification can still use the MAL rating.
    """
    if not anidb_id:
        return

    anidb_cache[str(anidb_id)] = {
        "cached_at": now_iso(),
        "source": "legacy_metadata",
        "metadata": normalize_anidb_metadata(metadata),
    }
    save_anidb_cache()


def get_anidb_cache_entry(anidb_id):
    if not anidb_id:
        return None

    return anidb_cache.get(str(anidb_id))


def parse_cached_at(value):
    if not value:
        return None

    try:
        return datetime.fromisoformat(str(value))
    except ValueError:
        return None


def is_recent_live_anidb_cache(anidb_id, max_age_days=ANIDB_LIVE_CACHE_TTL_DAYS):
    cached = get_anidb_cache_entry(anidb_id)

    if not cached or cached.get("source") != "live_http":
        return False

    cached_at = parse_cached_at(cached.get("cached_at"))

    if cached_at is None:
        return False

    age_seconds = (datetime.now() - cached_at).total_seconds()
    return age_seconds <= max_age_days * 24 * 60 * 60


def get_cached_anidb_metadata(anidb_id, rating=None):
    cached = get_anidb_cache_entry(anidb_id)

    if not cached:
        return None

    if "raw_tags" in cached:
        payload = {
            "episode_count": cached.get("episode_count"),
            "raw_tags": cached.get("raw_tags", []),
        }
        return classify_anidb_payload(payload, rating=rating)

    # Compatibility with cache entries created before raw tag payloads existed.
    return normalize_anidb_metadata(cached.get("metadata"))


def is_anidb_ban_response(root):
    if root.tag.lower() != "error":
        return False

    text = (root.text or "").strip().casefold()
    code = (root.get("code") or "").strip()

    return text == "banned" or code == "500"


def record_failed_id(
    mal_id,
    is_nsfw,
    stage,
    reason,
    status_code=None,
    retryable=True,
    idx=None,
    anidb_id=None,
):
    mal_id = int(mal_id)
    key = failure_key(mal_id, stage)
    previous = failed_registry.get(key, {})

    first_seen = previous.get("first_seen", now_iso())
    attempts = previous.get("attempts", 0) + 1

    failed_registry[key] = {
        "mal_id": mal_id,
        "is_nsfw": bool(is_nsfw),
        "stage": stage,
        "reason": str(reason),
        "status_code": status_code,
        "retryable": bool(retryable),
        "index": idx,
        "anidb_id": anidb_id,
        "attempts": attempts,
        "first_seen": first_seen,
        "last_seen": now_iso(),
    }

    save_failed_registry()
    build_stats["failed"] += 1

    write_log(
        f"FAILED | MAL {mal_id} | stage={stage} | "
        f"status={status_code} | retryable={retryable} | reason={reason}"
    )


def clear_failed_id(mal_id, stages=None):
    mal_id = int(mal_id)
    stages = set(stages) if stages else None
    removed = []

    for key, item in list(failed_registry.items()):
        if int(item.get("mal_id")) != mal_id:
            continue

        if stages and item.get("stage") not in stages:
            continue

        removed.append(item.get("stage"))
        failed_registry.pop(key)

    if removed:
        save_failed_registry()
        write_log(
            f"RECOVERED | MAL {mal_id} | removed stages={','.join(sorted(removed))}"
        )


def safe_request(
    url,
    timeout=15,
    headers=None,
    service="HTTP",
    entity_id=None,
    stage="request",
    max_attempts=3,
):
    headers = headers or {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=timeout
            )

        except requests.exceptions.RequestException as e:
            write_log(
                f"REQUEST_EXCEPTION | {service} | id={entity_id} | "
                f"stage={stage} | attempt={attempt}/{max_attempts} | {e}"
            )

            if attempt == max_attempts:
                return None

            time.sleep(5 * attempt)
            continue

        if response.status_code in RETRYABLE_STATUS_CODES and attempt < max_attempts:
            retry_after = response.headers.get("Retry-After")

            if retry_after and retry_after.isdigit():
                wait_seconds = int(retry_after)
            else:
                wait_seconds = 10 if response.status_code == 429 else 5 * attempt

            write_log(
                f"REQUEST_RETRY | {service} | id={entity_id} | "
                f"stage={stage} | HTTP {response.status_code} | "
                f"attempt={attempt}/{max_attempts} | sleep={wait_seconds}s"
            )
            time.sleep(wait_seconds)
            continue

        return response

    return None


def extract_names(items):
    """
    Extract only the 'name' field from API objects.
    """
    if not items:
        return ""

    return "|".join(
        x.get("name")
        for x in items
        if x.get("name")
    )


def split_pipe_values(value):
    if value is None:
        return []

    if isinstance(value, list):
        values = value
    elif pd.isna(value):
        return []
    else:
        values = str(value).split("|")

    return [str(item).strip() for item in values if str(item).strip()]


def merge_pipe_values(*values):
    merged = []
    seen = set()

    for value in values:
        for item in split_pipe_values(value):
            key = item.casefold()
            if key not in seen:
                seen.add(key)
                merged.append(item)

    return "|".join(merged)


def fixed_weight_values(value, weight=600):
    return "|".join(
        f"{item}:{weight}"
        for item in split_pipe_values(value)
    )


DEMOGRAPHIC_NAME_MAP = {
    "18 restricted": "18+",
    "18+": "18+",
    "adult": None,
    "josei": "Josei",
    "kids": "Kodomo",
    "kodomo": "Kodomo",
    "mina": "Mina",
    "seinen": "Seinen",
    "shoujo": "Shoujo",
    "shounen": "Shounen",
}


def normalize_demographic_name(value):
    key = str(value or "").strip().casefold()

    if not key:
        return None

    return DEMOGRAPHIC_NAME_MAP.get(key, str(value).strip().title())


def normalize_demographics(value):
    normalized = []

    for item in split_pipe_values(value):
        demographic = normalize_demographic_name(item)

        if demographic:
            normalized.append(demographic)

    return merge_pipe_values(normalized)


def normalize_dataset_columns(df):
    rename_columns = {}

    if "themes" in df.columns and "tags" not in df.columns:
        rename_columns["themes"] = "tags"

    if "explicit_genres" in df.columns and "explicit_tags" not in df.columns:
        rename_columns["explicit_genres"] = "explicit_tags"

    if "explicit_genre_weights" in df.columns and "explicit_tag_weights" not in df.columns:
        rename_columns["explicit_genre_weights"] = "explicit_tag_weights"

    if rename_columns:
        df = df.rename(columns=rename_columns)

    if "themes" in df.columns and "tags" in df.columns:
        df["tags"] = df.apply(
            lambda row: merge_pipe_values(row.get("tags"), row.get("themes")),
            axis=1
        )
        df = df.drop(columns=["themes"])

    if "explicit_genres" in df.columns:
        df["explicit_tags"] = df.apply(
            lambda row: merge_pipe_values(row.get("explicit_tags"), row.get("explicit_genres")),
            axis=1
        )
        df = df.drop(columns=["explicit_genres"])

    if "explicit_genre_weights" in df.columns:
        df["explicit_tag_weights"] = df.apply(
            lambda row: merge_pipe_values(row.get("explicit_tag_weights"), row.get("explicit_genre_weights")),
            axis=1
        )
        df = df.drop(columns=["explicit_genre_weights"])

    if "demographics" in df.columns:
        df["demographics"] = df["demographics"].apply(normalize_demographics)

    deprecated_columns = [
        col for col in ["relation_count", "is_nsfw", "recommendation_count"]
        if col in df.columns
    ]

    if deprecated_columns:
        df = df.drop(columns=deprecated_columns)

    return df


def clean_relations(relations):
    """
    Keep only important anime-to-anime relations.
    Output format: RelationType:MAL_ID|RelationType:MAL_ID
    """
    if not relations:
        return ""

    valid_relations = {
        "Prequel",
        "Sequel",
        "Side Story",
        "Alternative Version",
        "Summary",
        "Parent Story",
        "Character",
        "Spin-Off"
    }

    cleaned = []

    for rel in relations:
        relation_type = rel.get("relation")

        if relation_type not in valid_relations:
            continue

        for entry in rel.get("entry", []):
            if entry.get("type") != "anime":
                continue

            mal_id = entry.get("mal_id")

            if mal_id:
                cleaned.append(f"{relation_type}:{mal_id}")

    return "|".join(cleaned)


# =========================================================
# ANIDB ID AND METADATA EXTRACTION
# =========================================================

def extract_anidb_id(external_links):
    """
    Find AniDB external link and extract AID.
    """
    if not external_links:
        return None

    for ext in external_links:
        if ext.get("name") == "AniDB":
            url = ext.get("url", "")
            match = re.search(r"aid=(\d+)", url)

            if match:
                return int(match.group(1))

    return None


def normalize_anidb_name(name):
    return str(name).strip().casefold()


def has_explicit_rating(rating):
    rating_text = str(rating or "").casefold()
    return rating_text.startswith("r+") or rating_text.startswith("rx") or "hentai" in rating_text


def parse_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def tag_ancestor_names(tag_id, tags_by_id):
    ancestors = []
    seen = set()
    parent_id = tags_by_id.get(tag_id, {}).get("parent_id")

    while parent_id and parent_id not in seen:
        seen.add(parent_id)
        parent = tags_by_id.get(parent_id)

        if not parent:
            break

        ancestors.append(parent["name"])
        parent_id = parent.get("parent_id")

    return ancestors


# AniDB XML extraction lives in src/anidb_metadata_utils.py so Shoko cache
# imports and live AniDB calls save the same rich payload format.


def parse_anidb_tag_records(raw_tags, rating=None):
    """
    Split AniDB tag records into recommender tags, weighted metadata,
    explicit tags, and demographic labels.
    """
    tags_by_id = {
        int(tag["id"]): {
            "id": int(tag["id"]),
            "parent_id": tag.get("parent_id"),
            "name": tag.get("name", ""),
            "weight": parse_int(tag.get("weight"), default=0),
        }
        for tag in raw_tags or []
        if tag.get("id") is not None and tag.get("name")
    }

    parsed = {
        "tags": [],
        "tag_weights": [],
        "explicit_tags": [],
        "explicit_tag_weights": [],
        "demographics": [],
    }

    explicit_rating = has_explicit_rating(rating)

    for tag_id, tag in tags_by_id.items():
        name = tag["name"]
        name_key = normalize_anidb_name(name)
        parent_id = tag.get("parent_id")

        # Root branch labels are taxonomy containers, not anime features.
        if not parent_id:
            continue

        if name_key in DROPPED_ANIDB_TAGS:
            continue

        ancestors = tag_ancestor_names(tag_id, tags_by_id)
        lineage_keys = {name_key} | {normalize_anidb_name(item) for item in ancestors}

        if lineage_keys & IGNORED_ANIDB_BRANCHES:
            continue

        if DEMOGRAPHIC_BRANCH in lineage_keys:
            if name_key != DEMOGRAPHIC_BRANCH:
                demographic = normalize_demographic_name(name)
                if demographic:
                    parsed["demographics"].append(demographic)
            continue

        is_always_explicit = bool(lineage_keys & ALWAYS_EXPLICIT_BRANCHES)
        is_rating_gated_explicit = explicit_rating and bool(
            (lineage_keys & RATING_GATED_EXPLICIT_BRANCHES)
            or (name_key in RATING_GATED_EXPLICIT_TAGS)
        )

        if is_always_explicit or is_rating_gated_explicit:
            parsed["explicit_tags"].append(name)
            parsed["explicit_tag_weights"].append(f"{name}:{tag['weight']}")
            continue

        parsed["tags"].append(name)
        parsed["tag_weights"].append(f"{name}:{tag['weight']}")

    for key in parsed:
        parsed[key] = merge_pipe_values(parsed[key])

    return parsed


def parse_anidb_tags(root, rating=None):
    return parse_anidb_tag_records(
        extract_anidb_payload(root)["raw_tags"],
        rating=rating,
    )


def classify_anidb_payload(payload, rating=None):
    metadata = empty_anidb_metadata()
    metadata["episode_count"] = payload.get("episode_count")
    metadata.update(parse_anidb_tag_records(payload.get("raw_tags", []), rating=rating))
    return metadata


def write_anidb_metadata_logs(metadata, mal_id, anidb_id):
    if metadata.get("episode_count") is not None:
        build_stats["anidb_episode_fills"] += 1
        write_log(
            f"ANIDB_EPISODES | MAL {mal_id} | "
            f"AniDB {anidb_id} -> {metadata['episode_count']}"
        )

    if metadata.get("tags"):
        build_stats["anidb_tag_fills"] += 1
        write_log(
            f"ANIDB_TAGS | MAL {mal_id} | AniDB {anidb_id} | "
            f"tags={len(split_pipe_values(metadata['tags']))}"
        )

    if metadata.get("explicit_tags"):
        build_stats["anidb_explicit_fills"] += 1
        write_log(
            f"ANIDB_EXPLICIT | MAL {mal_id} | AniDB {anidb_id} | "
            f"explicit={metadata['explicit_tags']}"
        )

    if metadata.get("demographics"):
        build_stats["anidb_demographic_fills"] += 1
        write_log(
            f"ANIDB_DEMOGRAPHICS | MAL {mal_id} | AniDB {anidb_id} | "
            f"demographics={metadata['demographics']}"
        )


def anidb_cooldown_remaining_seconds():
    return max(0, int(ANIDB_COOLDOWN_UNTIL_TS - time.time()))


def set_anidb_cooldown(reason):
    global ANIDB_COOLDOWN_UNTIL_TS, ANIDB_LIVE_REQUESTS_IN_WINDOW

    ANIDB_COOLDOWN_UNTIL_TS = max(
        ANIDB_COOLDOWN_UNTIL_TS,
        time.time() + ANIDB_COOLDOWN_SECONDS,
    )
    ANIDB_LIVE_REQUESTS_IN_WINDOW = 0
    build_stats["anidb_cooldowns"] += 1

    cooldown_until = datetime.fromtimestamp(ANIDB_COOLDOWN_UNTIL_TS)
    write_log(
        f"ANIDB_COOLDOWN_SET | reason={reason} | "
        f"cooldown_seconds={ANIDB_COOLDOWN_SECONDS} | until={cooldown_until}"
    )


def throttle_anidb_request(mal_id=None, anidb_id=None, is_nsfw=False, failure_stage="anidb_metadata", live_reason="live_request"):
    global ANIDB_LIVE_REQUESTS_IN_WINDOW

    remaining = anidb_cooldown_remaining_seconds()

    if remaining > 0:
        build_stats["anidb_cooldown_skips"] += 1
        record_failed_id(
            mal_id or anidb_id,
            is_nsfw,
            failure_stage,
            f"AniDB cooldown active; retry after about {remaining}s; reason={live_reason}",
            status_code=500,
            retryable=True,
            anidb_id=anidb_id,
        )
        write_log(
            f"ANIDB_COOLDOWN_SKIP | MAL {mal_id} | AniDB {anidb_id} | "
            f"remaining_seconds={remaining} | reason={live_reason}"
        )
        return False

    wait_seconds = ANIDB_REQUEST_DELAY_SECONDS + random.uniform(
        0,
        ANIDB_REQUEST_JITTER_SECONDS,
    )
    time.sleep(wait_seconds)
    ANIDB_LIVE_REQUESTS_IN_WINDOW += 1
    build_stats["anidb_live_requests"] += 1
    return True


def handle_anidb_ban(mal_id, anidb_id, is_nsfw, status_code=500, failure_stage="anidb_metadata", live_reason="live_request"):
    build_stats["anidb_ban_resets"] += 1
    write_log(
        f"ANIDB_BANNED | MAL {mal_id} | AniDB {anidb_id} | "
        "cooldown set; dataset build continues"
    )
    record_failed_id(
        mal_id or anidb_id,
        is_nsfw,
        failure_stage,
        f"AniDB returned banned response; deferred; reason={live_reason}",
        status_code=status_code,
        retryable=True,
        anidb_id=anidb_id,
    )
    set_anidb_cooldown("banned_response")


def download_file(url, destination):
    destination = Path(destination)
    tmp_path = destination.with_suffix(destination.suffix + ".tmp")
    destination.parent.mkdir(parents=True, exist_ok=True)

    write_log(f"DOWNLOAD_START | {url} -> {destination}")

    with requests.get(
        url,
        stream=True,
        timeout=120,
        headers={"User-Agent": "Mozilla/5.0"},
    ) as response:
        response.raise_for_status()

        with open(tmp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    tmp_path.replace(destination)
    write_log(f"DOWNLOAD_COMPLETE | {destination}")


def zip_info_datetime(zip_info):
    try:
        return datetime(*zip_info.date_time).isoformat(timespec="seconds")
    except (TypeError, ValueError):
        return None


def populate_anidb_cache_from_xml_zip(
    zip_path=ANIDB_XML_CACHE_ZIP,
    source_url=ANIDB_XML_CACHE_URL,
    delete_zip_after_import=True,
):
    """
    Seed data/caches/anidb_metadata_cache.json from Shoko's Anime_HTTP.zip.
    The zip is treated as a disposable download; the durable artifact is the
    parsed AniDB cache.
    """
    zip_path = Path(zip_path)
    source_downloaded_at = now_iso()

    if not zip_path.exists():
        download_file(source_url, zip_path)

    imported = 0
    skipped_existing = 0
    parse_errors = 0
    anime_doc_pattern = re.compile(r"AnimeDoc_(\d+)\.xml$", re.IGNORECASE)

    try:
        with zipfile.ZipFile(zip_path) as zf:
            xml_infos = [
                info for info in zf.infolist()
                if anime_doc_pattern.search(info.filename)
            ]

            for info in xml_infos:
                match = anime_doc_pattern.search(info.filename)
                anidb_id = match.group(1)

                cached_entry = anidb_cache.get(anidb_id)
                if cached_entry and "raw_tags" in cached_entry:
                    has_rich_payload = all(
                        key in cached_entry
                        for key in [
                            "episode_summary",
                            "animation_work_creators",
                            "similar_anime",
                            "extra_metadata",
                        ]
                    )
                    if cached_entry.get("source") == "live_http" and has_rich_payload:
                        skipped_existing += 1
                        continue

                    if cached_entry.get("xml_file_modified_at") and has_rich_payload:
                        skipped_existing += 1
                        continue

                try:
                    root = ET.fromstring(zf.read(info))
                except ET.ParseError as e:
                    parse_errors += 1
                    write_log(f"SHOKO_XML_PARSE_ERROR | AniDB {anidb_id} | {e}")
                    continue

                if root.tag.lower() == "error":
                    parse_errors += 1
                    continue

                payload = extract_anidb_payload(root)
                payload["xml_file_modified_at"] = zip_info_datetime(info)
                payload["source_url"] = source_url
                payload["source_downloaded_at"] = source_downloaded_at

                cache_anidb_payload(
                    anidb_id,
                    payload,
                    source="shoko_xml_cache",
                    save=False,
                )
                imported += 1

                if imported % 500 == 0:
                    save_anidb_cache()
                    write_log(
                        f"SHOKO_XML_CACHE_PROGRESS | imported={imported} | "
                        f"skipped_existing={skipped_existing} | parse_errors={parse_errors}"
                    )

        build_stats["shoko_xml_cache_imported"] += imported
        build_stats["shoko_xml_cache_skipped_existing"] += skipped_existing
        build_stats["shoko_xml_cache_parse_errors"] += parse_errors

        if imported:
            save_anidb_cache()

        write_log(
            f"SHOKO_XML_CACHE_COMPLETE | imported={imported} | "
            f"skipped_existing={skipped_existing} | parse_errors={parse_errors} | "
            f"total_cache_entries={len(anidb_cache)}"
        )

        return {
            "imported": imported,
            "skipped_existing": skipped_existing,
            "parse_errors": parse_errors,
        }

    finally:
        if delete_zip_after_import and zip_path.exists():
            zip_path.unlink()
            write_log(f"DISPOSABLE_FILE_DELETED | {zip_path}")


def fetch_anidb_metadata(
    anidb_id,
    mal_id=None,
    is_nsfw=False,
    rating=None,
    allow_live=False,
    force_live=False,
    live_reason="missing_cache",
    failure_stage="anidb_metadata",
):
    """
    Return AniDB metadata from cache by default. Live HTTP is opt-in so the
    dataset build can continue even when AniDB is cooling down or unreliable.
    """
    metadata = empty_anidb_metadata()

    if not anidb_id:
        return metadata

    cached_metadata = get_cached_anidb_metadata(anidb_id, rating=rating)

    if cached_metadata is not None and not force_live:
        build_stats["anidb_cache_hits"] += 1
        write_log(f"ANIDB_CACHE_HIT | MAL {mal_id} | AniDB {anidb_id}")
        return cached_metadata

    if cached_metadata is not None and force_live and is_recent_live_anidb_cache(anidb_id):
        build_stats["anidb_cache_hits"] += 1
        write_log(
            f"ANIDB_RECENT_LIVE_CACHE_HIT | MAL {mal_id} | "
            f"AniDB {anidb_id} | reason={live_reason}"
        )
        return cached_metadata

    fallback_metadata = cached_metadata or metadata

    if not allow_live:
        return fallback_metadata

    if not throttle_anidb_request(
        mal_id=mal_id,
        anidb_id=anidb_id,
        is_nsfw=is_nsfw,
        failure_stage=failure_stage,
        live_reason=live_reason,
    ):
        return fallback_metadata

    url = (
        "http://api.anidb.net:9001/httpapi"
        f"?request=anime"
        f"&client={ANIDB_CLIENT}"
        f"&clientver={ANIDB_CLIENTVER}"
        f"&protover=1"
        f"&aid={anidb_id}"
    )

    response = safe_request(
        url,
        timeout=20,
        service="AniDB",
        entity_id=anidb_id,
        stage=failure_stage,
        max_attempts=1,
    )

    if response is None:
        record_failed_id(
            mal_id or anidb_id,
            is_nsfw,
            failure_stage,
            f"AniDB request exception after retries; reason={live_reason}",
            retryable=True,
            anidb_id=anidb_id,
        )
        return fallback_metadata

    if response.status_code != 200:
        try:
            error_root = ET.fromstring(response.content)
        except ET.ParseError:
            error_root = None

        if error_root is not None and is_anidb_ban_response(error_root):
            handle_anidb_ban(
                mal_id,
                anidb_id,
                is_nsfw,
                status_code=response.status_code,
                failure_stage=failure_stage,
                live_reason=live_reason,
            )
            return fallback_metadata

        retryable = response.status_code in RETRYABLE_STATUS_CODES
        record_failed_id(
            mal_id or anidb_id,
            is_nsfw,
            failure_stage,
            f"AniDB HTTP {response.status_code}; reason={live_reason}",
            status_code=response.status_code,
            retryable=retryable,
            anidb_id=anidb_id,
        )
        return fallback_metadata

    try:
        root = ET.fromstring(response.content)
    except ET.ParseError as e:
        record_failed_id(
            mal_id or anidb_id,
            is_nsfw,
            failure_stage,
            f"AniDB XML parse error: {e}; reason={live_reason}",
            retryable=True,
            anidb_id=anidb_id,
        )
        return fallback_metadata

    if is_anidb_ban_response(root):
        handle_anidb_ban(
            mal_id,
            anidb_id,
            is_nsfw,
            status_code=500,
            failure_stage=failure_stage,
            live_reason=live_reason,
        )
        return fallback_metadata

    if root.tag.lower() == "error":
        not_found = is_anidb_anime_not_found_response(root)
        record_failed_id(
            mal_id or anidb_id,
            is_nsfw,
            failure_stage,
            f"AniDB error response: {(root.text or '').strip()}; reason={live_reason}",
            status_code=root.get("code"),
            retryable=not not_found,
            anidb_id=anidb_id,
        )

        if not_found:
            record_permanent_http_skip_id(
                mal_id or anidb_id,
                status_code=None,
                reason="anidb_anime_not_found",
                anidb_id=anidb_id,
            )

        return fallback_metadata

    clear_failed_id(mal_id or anidb_id, stages=[failure_stage])

    payload = extract_anidb_payload(root)
    metadata = classify_anidb_payload(payload, rating=rating)
    write_anidb_metadata_logs(metadata, mal_id, anidb_id)
    cache_anidb_payload(anidb_id, payload, source="live_http")

    return metadata


# =========================================================
# RECOMMENDATIONS
# =========================================================

def fetch_recommendations(mal_id, is_nsfw=False):
    """
    Fetch every recommendation returned by Jikan.
    Output format: REC_MAL_ID:votes|REC_MAL_ID:votes
    """
    url = f"https://api.jikan.moe/v4/anime/{mal_id}/recommendations"

    response = safe_request(
        url,
        timeout=20,
        service="Jikan",
        entity_id=mal_id,
        stage="recommendations",
        max_attempts=3,
    )

    if response is None:
        record_failed_id(
            mal_id,
            is_nsfw,
            "recommendations",
            "Jikan recommendation request exception after retries",
            retryable=True,
        )
        return ""

    if response.status_code != 200:
        retryable = response.status_code in RETRYABLE_STATUS_CODES
        record_failed_id(
            mal_id,
            is_nsfw,
            "recommendations",
            f"Jikan recommendations HTTP {response.status_code}",
            status_code=response.status_code,
            retryable=retryable,
        )
        return ""

    clear_failed_id(mal_id, stages=["recommendations"])

    data = response.json().get("data", [])
    recommendations = []

    for rec in data:
        entry = rec.get("entry", {})
        rec_id = entry.get("mal_id")
        votes = rec.get("votes", 0)

        if rec_id:
            recommendations.append(f"{rec_id}:{votes}")

    if recommendations:
        build_stats["recommendation_fills"] += 1

    return "|".join(recommendations)


# =========================================================
# FILTERS AND ENTRY BUILDING
# =========================================================

def validate_anime(data, is_nsfw):
    anime_type = data.get("type")
    status = data.get("status")
    score = data.get("score")

    if anime_type not in VALID_TYPES:
        return False, f"invalid type: {anime_type}"

    if status not in VALID_STATUS:
        return False, f"invalid status: {status}"

    if score is None:
        return False, "no score"

    return True, "ok"


def build_entry(data, is_nsfw):
    titles = data.get("titles", [])
    default_title = None

    for t in titles:
        if t.get("type") == "Default":
            default_title = t.get("title")
            break

    aired_from = (
        data.get("aired", {})
            .get("prop", {})
            .get("from", {})
    )

    aired_year = aired_from.get("year")
    aired_month = aired_from.get("month")

    mal_id = data.get("mal_id")
    anidb_id = extract_anidb_id(data.get("external", []))
    relations = clean_relations(data.get("relations", []))
    effective_anidb_id = anidb_id
    anidb_metadata = fetch_anidb_metadata(
        anidb_id,
        mal_id=mal_id,
        is_nsfw=is_nsfw,
        rating=data.get("rating"),
        allow_live=False,
    )

    if anidb_id and is_empty_anidb_metadata(anidb_metadata):
        fallback_anidb_id = select_related_anidb_id_from_dataset(
            mal_id,
            relations,
            excluded_anidb_id=anidb_id,
        )

        if fallback_anidb_id:
            fallback_metadata = fetch_anidb_metadata(
                fallback_anidb_id,
                mal_id=mal_id,
                is_nsfw=is_nsfw,
                rating=data.get("rating"),
                allow_live=False,
            )

            if not is_empty_anidb_metadata(fallback_metadata):
                effective_anidb_id = fallback_anidb_id
                anidb_metadata = fallback_metadata
                write_log(
                    f"ANIDB_RELATED_FALLBACK_USED | MAL {mal_id} | "
                    f"AniDB {anidb_id} -> {fallback_anidb_id}"
                )

    episodes = data.get("episodes")
    is_currently_airing = data.get("status") == "Currently Airing"

    if episodes is None and is_currently_airing and anidb_id:
        if not is_recent_live_anidb_cache(anidb_id):
            refreshed_metadata = fetch_anidb_metadata(
                anidb_id,
                mal_id=mal_id,
                is_nsfw=is_nsfw,
                rating=data.get("rating"),
                allow_live=True,
                force_live=True,
                live_reason="missing_currently_airing_episode_count",
                failure_stage="anidb_episode_refresh",
            )

            if is_recent_live_anidb_cache(anidb_id):
                anidb_metadata = refreshed_metadata

    can_use_anidb_episode_count = (
        not is_currently_airing
        or is_recent_live_anidb_cache(anidb_id)
    )

    if (
        episodes is None
        and can_use_anidb_episode_count
        and anidb_metadata["episode_count"] is not None
    ):
        write_log(
            f"MISSING_EPISODES_FILLED | MAL {mal_id} | "
            f"{data.get('title')} | AniDB {anidb_id}"
        )
        episodes = anidb_metadata["episode_count"]

    mal_themes = extract_names(data.get("themes", []))
    mal_explicit_tags = extract_names(data.get("explicit_genres", []))

    tags = merge_pipe_values(mal_themes, anidb_metadata["tags"])
    tag_weights = merge_pipe_values(
        fixed_weight_values(mal_themes, weight=600),
        anidb_metadata["tag_weights"],
    )
    explicit_tags = merge_pipe_values(
        mal_explicit_tags,
        anidb_metadata["explicit_tags"],
    )
    explicit_tag_weights = merge_pipe_values(
        fixed_weight_values(mal_explicit_tags, weight=600),
        anidb_metadata["explicit_tag_weights"],
    )

    recommendations = fetch_recommendations(mal_id, is_nsfw=is_nsfw)
    time.sleep(0.8)

    mal_demographics = normalize_demographics(extract_names(data.get("demographics", [])))
    demographics = merge_pipe_values(anidb_metadata["demographics"], mal_demographics)

    return {
        "mal_id": mal_id,
        "anidb_id": effective_anidb_id,
        "url": data.get("url"),
        "image_url":
            data.get("images", {})
                .get("jpg", {})
                .get("large_image_url"),
        "title": default_title,
        "title_english": data.get("title_english"),
        "type": data.get("type"),
        "source": data.get("source"),
        "episodes": episodes,
        "status": data.get("status"),
        "duration": data.get("duration"),
        "rating": data.get("rating"),
        "score": data.get("score"),
        "scored_by": data.get("scored_by"),
        "rank": data.get("rank"),
        "popularity": data.get("popularity"),
        "members": data.get("members"),
        "favorites": data.get("favorites"),
        "synopsis": data.get("synopsis"),
        "aired_year": aired_year,
        "aired_month": aired_month,
        "season": data.get("season"),
        "genres": extract_names(data.get("genres", [])),
        "explicit_tags": explicit_tags,
        "explicit_tag_weights": explicit_tag_weights,
        "tags": tags,
        "tag_weights": tag_weights,
        "demographics": demographics,
        "studios": extract_names(data.get("studios", [])),
        "relations": relations,
        "recommendations": recommendations,
    }


def save_build_summary(results_count, elapsed_hours=None, completed=False):
    summary = {
        "updated_at": now_iso(),
        "completed": completed,
        "rows_collected": results_count,
        "failed_ids": len(failed_registry),
        "invalid_type_ids": len(invalid_type_registry),
        "permanent_http_skip_ids": len(permanent_http_skip_registry),
        "anidb_cache_entries": len(anidb_cache),
        "stats": build_stats,
    }

    if elapsed_hours is not None:
        summary["hours"] = round(elapsed_hours, 4)

    atomic_write_json(BUILD_SUMMARY_FILE, summary)


def save_outputs(results, checkpoint_index=None, elapsed_hours=None, completed=False):
    df_temp = normalize_dataset_columns(pd.DataFrame(results))
    atomic_write_csv(df_temp, OUTPUT_CSV)

    atomic_write_json(
        OUTPUT_JSON,
        df_temp.to_dict(orient="records")
    )

    if checkpoint_index is not None:
        atomic_write_json(CHECKPOINT_FILE, {"index": checkpoint_index})

    save_failed_registry()
    save_invalid_type_registry()
    save_permanent_http_skip_registry()
    save_build_summary(
        len(df_temp),
        elapsed_hours=elapsed_hours,
        completed=completed,
    )


def backfill_missing_anidb_cache_from_dataset(limit=None):
    if not OUTPUT_CSV.exists():
        write_log("ANIDB_POST_BUILD_BACKFILL_SKIPPED | dataset CSV missing")
        return {"attempted": 0, "cached": 0}

    df = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))

    if "anidb_id" not in df.columns:
        write_log("ANIDB_POST_BUILD_BACKFILL_SKIPPED | anidb_id column missing")
        return {"attempted": 0, "cached": 0}

    candidates = df[df["anidb_id"].notna()].copy()
    candidates["anidb_id"] = candidates["anidb_id"].astype(int)
    candidates = candidates[
        ~candidates["anidb_id"].astype(str).isin(anidb_cache.keys())
    ]

    if "popularity" in candidates.columns:
        candidates = candidates.sort_values("popularity", na_position="last")

    attempted = 0
    cached = 0

    for _, row in candidates.iterrows():
        if limit is not None and attempted >= limit:
            break

        if anidb_cooldown_remaining_seconds() > 0:
            write_log("ANIDB_POST_BUILD_BACKFILL_PAUSED | cooldown active")
            break

        anidb_id = int(row["anidb_id"])
        mal_id = int(row["mal_id"])

        if is_known_permanent_anidb_skip(mal_id, anidb_id):
            write_log(
                f"ANIDB_POST_BUILD_BACKFILL_SKIPPED_PERMANENT | "
                f"MAL {mal_id} | AniDB {anidb_id}"
            )
            continue

        attempted += 1
        before_cached = str(anidb_id) in anidb_cache

        fetch_anidb_metadata(
            anidb_id,
            mal_id=mal_id,
            is_nsfw=False,
            rating=row.get("rating"),
            allow_live=True,
            force_live=True,
            live_reason="post_build_missing_anidb_cache_backfill",
            failure_stage="anidb_metadata",
        )

        if not before_cached and str(anidb_id) in anidb_cache:
            cached += 1
            continue

        fallback_anidb_id = select_related_anidb_id_from_dataset(
            mal_id,
            row.get("relations"),
            excluded_anidb_id=anidb_id,
        )

        if fallback_anidb_id:
            if mal_id in permanent_http_skip_registry:
                permanent_http_skip_registry.pop(mal_id)
                save_permanent_http_skip_registry()
                write_log(
                    f"PERMANENT_SKIP_REMOVED_RELATED_ANIDB_FALLBACK | "
                    f"MAL {mal_id} | fallback_anidb_id={fallback_anidb_id}"
                )

            if str(fallback_anidb_id) not in anidb_cache:
                before_fallback_cached = str(fallback_anidb_id) in anidb_cache
                fetch_anidb_metadata(
                    fallback_anidb_id,
                    mal_id=mal_id,
                    is_nsfw=False,
                    rating=row.get("rating"),
                    allow_live=True,
                    force_live=True,
                    live_reason="post_build_related_anidb_fallback",
                    failure_stage="anidb_metadata",
                )

                if not before_fallback_cached and str(fallback_anidb_id) in anidb_cache:
                    cached += 1

        elif failure_key(mal_id, "anidb_metadata") in failed_registry:
            failed_item = failed_registry[failure_key(mal_id, "anidb_metadata")]
            if "Anime not found" in str(failed_item.get("reason")):
                record_permanent_http_skip_id(
                    mal_id,
                    status_code=None,
                    reason="anidb_anime_not_found",
                    anidb_id=anidb_id,
                )

    write_log(
        f"ANIDB_POST_BUILD_BACKFILL_COMPLETE | attempted={attempted} | cached={cached}"
    )
    return {"attempted": attempted, "cached": cached}


def process_mal_id(mal_id, is_nsfw, idx=None, recovery=False):
    try:
        url = f"https://api.jikan.moe/v4/anime/{mal_id}/full"

        response = safe_request(
            url,
            timeout=20,
            service="Jikan",
            entity_id=mal_id,
            stage="anime_full",
            max_attempts=3,
        )

        if response is None:
            record_failed_id(
                mal_id,
                is_nsfw,
                "anime_full",
                "Jikan anime/full request exception after retries",
                retryable=True,
                idx=idx,
            )
            return None

        if response.status_code != 200:
            retryable = response.status_code in RETRYABLE_STATUS_CODES

            if response.status_code in PERMANENT_HTTP_SKIP_STATUS_CODES:
                record_permanent_http_skip_id(
                    mal_id,
                    response.status_code,
                    idx=idx,
                )

            if retryable:
                record_failed_id(
                    mal_id,
                    is_nsfw,
                    "anime_full",
                    f"Jikan anime/full HTTP {response.status_code}",
                    status_code=response.status_code,
                    retryable=True,
                    idx=idx,
                )
            else:
                write_log(
                    f"SKIPPED | MAL {mal_id} | HTTP {response.status_code} | retryable=False"
                )

            return None

        data = response.json().get("data")

        if not data:
            record_failed_id(
                mal_id,
                is_nsfw,
                "anime_full",
                "Jikan returned no data payload",
                retryable=True,
                idx=idx,
            )
            return None

        is_valid, filter_reason = validate_anime(data, is_nsfw)

        if not is_valid:
            build_stats["filtered"] += 1
            clear_failed_id(mal_id)

            if filter_reason.startswith("invalid type:"):
                record_invalid_type_id(
                    mal_id,
                    is_nsfw,
                    data.get("type"),
                    idx=idx,
                )

            write_log(f"FILTERED | MAL {mal_id} | {filter_reason}")
            return None

        entry = build_entry(data, is_nsfw)
        clear_failed_id(mal_id, stages=["anime_full"])

        write_log(
            f"ADDED | MAL {mal_id} | {entry['title']} | {entry['type']} | "
            f"Score {entry['score']} | Members {entry['members']} | "
            f"Tags {entry['tags']}"
        )

        return entry

    except Exception as e:
        record_failed_id(
            mal_id,
            is_nsfw,
            "process_mal_id",
            f"Unexpected processing error: {type(e).__name__}: {e}",
            retryable=True,
            idx=idx,
        )
        write_log(
            f"ERROR | MAL {mal_id} | unexpected {type(e).__name__}: {e}"
        )
        return None


failed_registry = load_failed_registry()
invalid_type_registry = load_invalid_type_registry()
permanent_http_skip_registry = load_permanent_http_skip_registry()
anidb_cache = load_anidb_cache()

In [13]:
# =========================================================
# CHECKPOINT LOADING
# =========================================================

if CHECKPOINT_FILE.exists() and OUTPUT_CSV.exists():

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)

    start_index = checkpoint["index"]

    print(f"Resuming from index {start_index}")

else:

    start_index = 0

    if CHECKPOINT_FILE.exists() and not OUTPUT_CSV.exists():
        print(
            "Checkpoint exists but processed dataset is missing. "
            "Starting from index 0."
        )

Resuming from index 30013


In [14]:
# =========================================================
# ANIDB XML CACHE REFRESH
# =========================================================

populate_anidb_cache_from_xml_zip()


[2026-05-16 20:06:38.310630] DOWNLOAD_START | https://files.shokoanime.com/files/shoko-server/other/Anime_HTTP.zip -> C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\Anime_HTTP.zip
[2026-05-16 20:06:43.591739] DOWNLOAD_COMPLETE | C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\Anime_HTTP.zip
[2026-05-16 20:06:43.680961] SHOKO_XML_CACHE_COMPLETE | imported=0 | skipped_existing=15638 | parse_errors=0 | total_cache_entries=15638
[2026-05-16 20:06:43.692521] DISPOSABLE_FILE_DELETED | C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\Anime_HTTP.zip


{'imported': 0, 'skipped_existing': 15638, 'parse_errors': 0}

In [15]:
# =========================================================
# MAIN LOOP
# =========================================================

results = []
existing_ids = set()

if OUTPUT_CSV.exists():
    existing_df = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))
    results = existing_df.to_dict(orient="records")
    existing_ids = set(existing_df["mal_id"].dropna().astype(int).tolist())
    print(f"Loaded existing dataset: {len(results)} rows")

write_section_log(
    "DATASET BUILD START",
    {
        "SFW IDs": len(sfw_ids),
        "NSFW IDs": len(nsfw_ids),
        "Total candidate IDs": len(all_entries),
        "Resume index": start_index,
        "Existing processed rows": len(results),
        "Existing failed IDs": len(failed_registry),
        "Existing invalid-type filtered IDs": len(invalid_type_registry),
        "Existing permanent HTTP skip IDs": len(permanent_http_skip_registry),
        "Existing AniDB cache entries": len(anidb_cache),
        "AniDB request delay seconds": ANIDB_REQUEST_DELAY_SECONDS,
        "AniDB request jitter seconds": ANIDB_REQUEST_JITTER_SECONDS,
        "AniDB cooldown seconds": ANIDB_COOLDOWN_SECONDS,
        "AniDB live cache TTL days": ANIDB_LIVE_CACHE_TTL_DAYS,
        "Shoko XML cache zip": ANIDB_XML_CACHE_ZIP,
        "Shoko XML cache URL": ANIDB_XML_CACHE_URL,
        "Valid types": sorted(VALID_TYPES),
        "Valid status": sorted(VALID_STATUS),
        "Output CSV": OUTPUT_CSV,
        "Failed IDs file": FAILED_IDS_FILE,
        "Invalid type IDs file": FILTERED_INVALID_TYPES_FILE,
        "Permanent HTTP skip file": PERMANENT_HTTP_SKIP_FILE,
        "Build summary file": BUILD_SUMMARY_FILE,
    }
)

start_time = time.time()

for idx in range(start_index, len(all_entries)):
    mal_id, is_nsfw = all_entries[idx]
    build_stats["processed"] += 1

    if int(mal_id) in existing_ids:
        build_stats["skipped_existing"] += 1
        continue

    if is_known_invalid_type(mal_id):
        build_stats["skipped_invalid_type"] += 1
        write_log(f"SKIPPED_INVALID_TYPE_CACHE | MAL {mal_id}")
        continue

    if is_known_permanent_http_skip(mal_id):
        build_stats["skipped_permanent_http"] += 1
        write_log(f"SKIPPED_PERMANENT_HTTP_CACHE | MAL {mal_id}")
        continue

    entry = process_mal_id(
        mal_id,
        is_nsfw,
        idx=idx,
        recovery=False,
    )

    if entry:
        results.append(entry)
        existing_ids.add(int(mal_id))
        build_stats["added"] += 1

    if build_stats["processed"] % 5 == 0:
        save_outputs(results, checkpoint_index=idx + 1)
        write_log(
            f"CHECKPOINT | index={idx + 1} | MAL {mal_id} | "
            f"rows={len(results)} | failed={len(failed_registry)}"
        )

    time.sleep(0.8)

# =========================================================
# FINAL SAVE
# =========================================================

elapsed = time.time() - start_time
hours = elapsed / 3600

save_outputs(
    results,
    checkpoint_index=len(all_entries),
    elapsed_hours=hours,
    completed=True,
)

df_final = normalize_dataset_columns(pd.DataFrame(results))

print("\n=========================================")
print("DATASET BUILD COMPLETE")
print("=========================================")
print(f"Rows collected: {len(df_final)}")
print(f"Failed IDs still queued: {len(failed_registry)}")
print(f"Output: {OUTPUT_CSV}")
print(f"Failed IDs file: {FAILED_IDS_FILE}")
print(f"Elapsed time: {hours:.2f} hours")
print("=========================================")

write_section_log(
    "DATASET BUILD COMPLETE",
    {
        "Rows collected": len(df_final),
        "Failed IDs still queued": len(failed_registry),
        "Processed loop iterations": build_stats["processed"],
        "Added rows this run": build_stats["added"],
        "Filtered this run": build_stats["filtered"],
        "Skipped existing this run": build_stats["skipped_existing"],
        "Skipped invalid type cache this run": build_stats["skipped_invalid_type"],
        "Skipped permanent HTTP cache this run": build_stats["skipped_permanent_http"],
        "AniDB episode fills": build_stats["anidb_episode_fills"],
        "AniDB tag fills": build_stats["anidb_tag_fills"],
        "AniDB explicit fills": build_stats["anidb_explicit_fills"],
        "AniDB demographic fills": build_stats["anidb_demographic_fills"],
        "AniDB cache hits": build_stats["anidb_cache_hits"],
        "AniDB live requests": build_stats["anidb_live_requests"],
        "AniDB cooldowns": build_stats["anidb_cooldowns"],
        "AniDB cooldown skips": build_stats["anidb_cooldown_skips"],
        "AniDB ban resets": build_stats["anidb_ban_resets"],
        "Shoko XML cache imported": build_stats["shoko_xml_cache_imported"],
        "Shoko XML cache skipped existing": build_stats["shoko_xml_cache_skipped_existing"],
        "Shoko XML cache parse errors": build_stats["shoko_xml_cache_parse_errors"],
        "Recommendation fills": build_stats["recommendation_fills"],
        "Hours": f"{hours:.2f}",
    }
)

Loaded existing dataset: 16670 rows
[2026-05-16 20:06:44.184802] ========================================================================
[2026-05-16 20:06:44.185216] DATASET BUILD START
[2026-05-16 20:06:44.185478] SFW IDs: 28390
[2026-05-16 20:06:44.185809] NSFW IDs: 1623
[2026-05-16 20:06:44.186098] Total candidate IDs: 30013
[2026-05-16 20:06:44.186358] Resume index: 30013
[2026-05-16 20:06:44.186714] Existing processed rows: 16670
[2026-05-16 20:06:44.186979] Existing failed IDs: 18
[2026-05-16 20:06:44.187227] Existing invalid-type filtered IDs: 5235
[2026-05-16 20:06:44.187426] Existing permanent HTTP skip IDs: 10
[2026-05-16 20:06:44.187623] Existing AniDB cache entries: 15638
[2026-05-16 20:06:44.187813] AniDB request delay seconds: 4
[2026-05-16 20:06:44.188007] AniDB request jitter seconds: 0.5
[2026-05-16 20:06:44.188197] AniDB cooldown seconds: 1800
[2026-05-16 20:06:44.188396] AniDB live cache TTL days: 30
[2026-05-16 20:06:44.188634] Shoko XML cache zip: C:\Users\CHAMPUX

In [18]:
# =========================================================
# MANUAL POST-BUILD ANIDB CACHE BACKFILL
# =========================================================
# Legacy broad backfill. Prefer the targeted live-repair cells below because
# they spend AniDB calls first on episode gaps, then on popular metadata gaps.

#backfill_missing_anidb_cache_from_dataset(limit=100)


[2026-05-16 20:07:27.699063] ANIDB_EPISODES | MAL 62568 | AniDB 19614 -> 13
[2026-05-16 20:07:35.266432] ANIDB_EPISODES | MAL 62601 | AniDB 19625 -> 8
[2026-05-16 20:07:35.267011] ANIDB_TAGS | MAL 62601 | AniDB 19625 | tags=2
[2026-05-16 20:07:43.171136] ANIDB_EPISODES | MAL 62896 | AniDB 19701 -> 1
[2026-05-16 20:07:43.171936] ANIDB_TAGS | MAL 62896 | AniDB 19701 | tags=24
[2026-05-16 20:07:50.792364] ANIDB_EPISODES | MAL 62604 | AniDB 19628 -> 12
[2026-05-16 20:07:50.792905] ANIDB_TAGS | MAL 62604 | AniDB 19628 | tags=13
[2026-05-16 20:07:50.793134] ANIDB_DEMOGRAPHICS | MAL 62604 | AniDB 19628 | demographics=Seinen
[2026-05-16 20:07:58.305871] ANIDB_EPISODES | MAL 63375 | AniDB 19861 -> 12
[2026-05-16 20:07:58.306339] ANIDB_TAGS | MAL 63375 | AniDB 19861 | tags=12
[2026-05-16 20:08:05.731496] ANIDB_EPISODES | MAL 63019 | AniDB 19743 -> 20
[2026-05-16 20:08:05.732000] ANIDB_TAGS | MAL 63019 | AniDB 19743 | tags=2
[2026-05-16 20:08:13.479033] ANIDB_EPISODES | MAL 62913 | AniDB 19713 ->

{'attempted': 81, 'cached': 81}

# =========================================================
# OPTIONAL POST-BUILD ANIDB LIVE REPAIR
# =========================================================

AniDB live calls are valuable and can trigger cooldowns. Use these cells after the main dataset build:

1. repair missing or zero episode counts first
2. repair tag/demographic/explicit metadata gaps by MAL popularity

Every successful AniDB response is saved immediately to `data/caches/anidb_metadata_cache.json`, and failures continue to be recorded through the dataset builder's normal failed-request registry.

In [19]:
def is_missing_text_for_repair(value):
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except TypeError:
        pass
    text = str(value).strip()
    return text == "" or text.lower() in {"nan", "none", "null"}


def anidb_cached_episode_count(anidb_id):
    if pd.isna(anidb_id):
        return None
    payload = anidb_cache.get(str(int(anidb_id)), {})
    return payload.get("episode_count")


def anidb_payload_needs_metadata_refresh(row):
    tags_missing = is_missing_text_for_repair(row.get("tags"))
    demographics_missing = is_missing_text_for_repair(row.get("demographics"))
    explicit_missing = is_missing_text_for_repair(row.get("explicit_tags")) and has_explicit_rating(row.get("rating"))
    studios_missing = is_missing_text_for_repair(row.get("studios"))
    return tags_missing or demographics_missing or explicit_missing or studios_missing


def build_anidb_live_repair_candidates():
    if not OUTPUT_CSV.exists():
        print("Dataset CSV is missing. Run the main build first.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_repair = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))

    for required_column in ["tags", "explicit_tags", "demographics", "studios"]:
        if required_column not in df_repair.columns:
            df_repair[required_column] = ""

    episode_gap = (
        df_repair["episodes"].isna()
        | (pd.to_numeric(df_repair["episodes"], errors="coerce").fillna(-1) == 0)
    )
    episode_candidates = df_repair.loc[
        episode_gap & df_repair["anidb_id"].notna(),
        ["mal_id", "anidb_id", "title", "type", "status", "episodes", "rating", "popularity"],
    ].copy()
    episode_candidates["cached_episode_count"] = episode_candidates["anidb_id"].apply(
        anidb_cached_episode_count
    )
    cached_episode_count = pd.to_numeric(episode_candidates["cached_episode_count"], errors="coerce")
    episode_candidates = episode_candidates.loc[
        cached_episode_count.isna() | (cached_episode_count <= 0)
    ].copy()
    episode_candidates = episode_candidates.sort_values("popularity", na_position="last")

    metadata_mask = df_repair.apply(anidb_payload_needs_metadata_refresh, axis=1)
    metadata_candidates = df_repair.loc[
        metadata_mask & df_repair["anidb_id"].notna(),
        [
            "mal_id",
            "anidb_id",
            "title",
            "type",
            "rating",
            "popularity",
            "tags",
            "explicit_tags",
            "demographics",
            "studios",
        ],
    ].copy()
    metadata_candidates["needs_tags"] = metadata_candidates["tags"].apply(is_missing_text_for_repair)
    metadata_candidates["needs_explicit_tags"] = (
        metadata_candidates["explicit_tags"].apply(is_missing_text_for_repair)
        & metadata_candidates["rating"].apply(has_explicit_rating)
    )
    metadata_candidates["needs_demographics"] = metadata_candidates["demographics"].apply(is_missing_text_for_repair)
    metadata_candidates["needs_studios"] = metadata_candidates["studios"].apply(is_missing_text_for_repair)
    metadata_candidates = metadata_candidates.sort_values("popularity", na_position="last")


    duration_gap = df_repair["duration"].isna() if "duration" in df_repair.columns else pd.Series(False, index=df_repair.index)
    runtime_gap = (
        df_repair["total_watch_minutes"].isna()
        if "total_watch_minutes" in df_repair.columns
        else pd.Series(False, index=df_repair.index)
    )
    duration_candidates = df_repair.loc[
        (duration_gap | runtime_gap) & df_repair["anidb_id"].notna(),
        ["mal_id", "anidb_id", "title", "type", "status", "episodes", "duration", "total_watch_minutes", "rating", "popularity"],
    ].copy()
    duration_candidates["needs_duration"] = duration_gap.loc[duration_candidates.index].to_numpy()
    duration_candidates["needs_total_watch_minutes"] = runtime_gap.loc[duration_candidates.index].to_numpy()
    duration_candidates = duration_candidates.sort_values("popularity", na_position="last")

    airing_mask = (
        df_repair["status"].fillna("").astype(str).str.casefold().eq("currently airing")
    )
    airing_candidates = df_repair.loc[
        airing_mask & df_repair["anidb_id"].notna(),
        ["mal_id", "anidb_id", "title", "type", "status", "episodes", "duration", "total_watch_minutes", "rating", "popularity"],
    ].copy()
    airing_candidates = airing_candidates.sort_values("popularity", na_position="last")

    need_columns = ["needs_explicit_tags", "needs_studios", "needs_tags", "needs_demographics"]
    missing_counts = {
        column: int(metadata_candidates[column].sum())
        for column in need_columns
    }
    metadata_candidates["scarcity_priority"] = metadata_candidates[need_columns].apply(
        lambda row: min(
            [
                missing_counts[column]
                for column, needs_value in row.items()
                if bool(needs_value) and missing_counts[column] > 0
            ]
            or [999999]
        ),
        axis=1,
    )
    metadata_candidates["need_count"] = metadata_candidates[need_columns].sum(axis=1)
    metadata_candidates = metadata_candidates.sort_values(
        ["scarcity_priority", "popularity"],
        na_position="last",
    )

    return episode_candidates, duration_candidates, airing_candidates, metadata_candidates


episode_live_candidates, duration_live_candidates, airing_live_candidates, metadata_live_candidates = build_anidb_live_repair_candidates()

print(f"Episode live candidates: {len(episode_live_candidates)}")
display(episode_live_candidates)

print(f"Duration/runtime live candidates: {len(duration_live_candidates)}")
display(duration_live_candidates.head(100))

print(f"Currently-airing refresh candidates: {len(airing_live_candidates)}")
display(airing_live_candidates.head(100))

print(f"Metadata live candidates: {len(metadata_live_candidates)}")
display(metadata_live_candidates.head(100))

Episode live candidates: 8


,mal_id,anidb_id,title,type,status,episodes,rating,popularity,cached_episode_count
12585,61469,19287.0,Steel Ball Run: JoJo no Kimyou na Bouken,ONA,Currently Airing,0.0,R - 17+ (violence & profanity),1371,0
12355,59970,18884.0,Tensei shitara Slime Datta Ken 4th Season,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,1457,0
12510,60852,19139.0,Koori no Jouheki,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,2811,0
12274,59443,18791.0,Reincarnation no Kaben,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,3781,0
12367,60055,18900.0,Yozakura-san Chi no Daisakusen 2nd Season,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,4252,0
12262,59393,18772.0,Niwatori Fighter,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,4333,0
12559,61269,19230.0,Digimon Beatbreak,TV,Currently Airing,0.0,PG-13 - Teens 13 or older,4927,0
12383,60153,18936.0,Rilakkuma,TV,Currently Airing,0.0,PG - Children,11730,0


Metadata live candidates: 5447


,mal_id,anidb_id,title,type,rating,popularity,tags,explicit_tags,demographics,needs_tags,needs_explicit_tags,needs_demographics
7316,32281,11829.0,Kimi no Na wa.,Movie,PG-13 - Teens 13 or older,12,breast fondling|amnesia|time travel|science fi...,NaN,,False,False,True
7065,31240,11370.0,Re:Zero kara Hajimeru Isekai Seikatsu,TV,R - 17+ (violence & profanity),23,Isekai|Psychological|Time Travel|magic|bishouj...,NaN,,False,False,True
2828,4224,5909.0,Toradora!,TV,PG-13 - Teens 13 or older,26,Love Polygon|School|baseball|horny nosebleed|s...,NaN,,False,False,True
8737,37510,13939.0,Mob Psycho 100 II,TV,PG-13 - Teens 13 or older,73,super power|nearly almighty protagonist|cast,NaN,,False,False,True
6491,28121,10894.0,Dungeon ni Deai wo Motomeru no wa Machigatteir...,TV,PG-13 - Teens 13 or older,77,juujin|elf|magic|dragon|deity|collateral damag...,NaN,,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
10320,47164,16032.0,Dungeon ni Deai wo Motomeru no wa Machigatteir...,TV,R - 17+ (violence & profanity),663,fantasy|speculative fiction,NaN,,False,False,True
4543,10490,8348.0,Blood-C,TV,R - 17+ (violence & profanity),665,Gore|School|Vampire|undead|high school|amnesia...,NaN,,False,False,True
10372,48417,16101.0,Maou Gakuin no Futekigousha II: Shijou Saikyou...,TV,R - 17+ (violence & profanity),667,Reincarnation|School|demon|fantasy|nearly almi...,NaN,,False,False,True
5151,15037,9351.0,Corpse Party: Tortured Souls - Bougyakusareta ...,OVA,R - 17+ (violence & profanity),669,Gore|undead|high school|ghost|violence|pantsu|...,BDSM|pornography,,False,False,True


In [20]:
def run_anidb_live_repair_candidates(candidates, label, limit=None):
    if candidates is None or candidates.empty:
        print(f"{label}: no candidates")
        return {"attempted": 0, "cached": 0}

    selected = candidates if limit is None else candidates.head(limit)
    attempted = 0
    cached = 0
    total = len(selected)

    for position, (_, row) in enumerate(selected.iterrows(), start=1):
        anidb_id = int(row["anidb_id"])
        mal_id = int(row["mal_id"])

        if anidb_cooldown_remaining_seconds() > 0:
            remaining = anidb_cooldown_remaining_seconds()
            print(f"[{position}/{total}] {label} | cooldown active | remaining_seconds={remaining}")
            write_log(f"ANIDB_LIVE_REPAIR_PAUSED | label={label} | remaining_seconds={remaining}")
            break

        print(
            f"[{position}/{total}] {label} | MAL {mal_id} | AniDB {anidb_id} | "
            f"popularity={row.get('popularity')} | request_start",
            flush=True,
        )

        before_cached = str(anidb_id) in anidb_cache
        before_cached_at = (anidb_cache.get(str(anidb_id), {}) or {}).get("cached_at")

        attempted += 1
        fetch_anidb_metadata(
            anidb_id,
            mal_id=mal_id,
            is_nsfw=str(row.get("rating")) == "Rx - Hentai",
            rating=row.get("rating"),
            allow_live=True,
            force_live=True,
            live_reason=f"post_build_{label}",
            failure_stage="anidb_metadata",
        )

        after_payload = anidb_cache.get(str(anidb_id), {}) or {}
        after_cached_at = after_payload.get("cached_at")
        if str(anidb_id) in anidb_cache and (not before_cached or after_cached_at != before_cached_at):
            cached += 1
            save_anidb_cache()
            print(
                f"[{position}/{total}] {label} | MAL {mal_id} | AniDB {anidb_id} | "
                f"saved | episode_count={after_payload.get('episode_count')} | "
                f"raw_tags={len(after_payload.get('raw_tags', []) or [])}",
                flush=True,
            )
        else:
            print(f"[{position}/{total}] {label} | MAL {mal_id} | AniDB {anidb_id} | no_update")

    print(f"{label}: attempted={attempted}, cached={cached}")
    return {"attempted": attempted, "cached": cached}


# Step 1: spend AniDB calls on episode gaps first.
episode_repair_result = run_anidb_live_repair_candidates(
    episode_live_candidates,
    label="episode_repair",
    limit=None,
)

[1/8] episode_repair | MAL 61469 | AniDB 19287 | popularity=1371 | request_start
[2026-05-16 20:18:59.708116] ANIDB_EPISODES | MAL 61469 | AniDB 19287 -> 1
[1/8] episode_repair | MAL 61469 | AniDB 19287 | saved | episode_count=1 | raw_tags=6
[2/8] episode_repair | MAL 59970 | AniDB 18884 | popularity=1457 | request_start
[2026-05-16 20:19:11.227320] ANIDB_EPISODES | MAL 59970 | AniDB 18884 -> 12
[2026-05-16 20:19:11.227875] ANIDB_TAGS | MAL 59970 | AniDB 18884 | tags=3
[2026-05-16 20:19:11.228102] ANIDB_DEMOGRAPHICS | MAL 59970 | AniDB 18884 | demographics=Shounen
[2/8] episode_repair | MAL 59970 | AniDB 18884 | saved | episode_count=12 | raw_tags=11
[3/8] episode_repair | MAL 60852 | AniDB 19139 | popularity=2811 | request_start
[2026-05-16 20:19:22.498280] ANIDB_EPISODES | MAL 60852 | AniDB 19139 -> 12
[2026-05-16 20:19:22.499027] ANIDB_TAGS | MAL 60852 | AniDB 19139 | tags=5
[3/8] episode_repair | MAL 60852 | AniDB 19139 | saved | episode_count=12 | raw_tags=16
[4/8] episode_repair 

In [ ]:
# Step 2: then spend AniDB calls on popular metadata gaps.
# metadata_repair_result = run_anidb_live_repair_candidates(
#     metadata_live_candidates,
#     label="metadata_repair",
#     limit=100,
# )

# =========================================================
# RETRY FAILED IDS
# =========================================================
# Run this cell after the main build if failed_api_requests.json contains retryable failures.

In [23]:
# =========================================================
# RETRY FAILED IDS USING THE SAME FILTERS
# =========================================================
ANIDB_LIVE_REQUESTS_IN_WINDOW = 0
ANIDB_COOLDOWN_UNTIL_TS = 0
failed_registry = load_failed_registry()
invalid_type_registry = load_invalid_type_registry()
permanent_http_skip_registry = load_permanent_http_skip_registry()
retry_items_by_mal_id = {}

for item in failed_registry.values():
    if not item.get("retryable", True):
        continue

    mal_id = int(item["mal_id"])
    previous = retry_items_by_mal_id.get(mal_id)

    if previous is None or item.get("last_seen", "") > previous.get("last_seen", ""):
        retry_items_by_mal_id[mal_id] = item

retry_items = list(retry_items_by_mal_id.values())

print(f"Retryable failed IDs: {len(retry_items)}")

if OUTPUT_CSV.exists():
    df_existing = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))
else:
    df_existing = pd.DataFrame()

results_by_id = {}

if not df_existing.empty:
    for row in df_existing.to_dict(orient="records"):
        results_by_id[int(row["mal_id"])] = row

recovered = 0

for idx, item in enumerate(sorted(retry_items, key=lambda x: int(x["mal_id"]))):
    mal_id = int(item["mal_id"])
    is_nsfw = bool(item.get("is_nsfw", False))

    if is_known_invalid_type(mal_id):
        write_log(f"RETRY_SKIPPED_INVALID_TYPE_CACHE | MAL {mal_id}")
        continue

    if is_known_permanent_http_skip(mal_id):
        write_log(f"RETRY_SKIPPED_PERMANENT_HTTP_CACHE | MAL {mal_id}")
        continue

    write_log(
        f"RETRY | {idx + 1}/{len(retry_items)} | MAL {mal_id} | "
        f"previous_stage={item.get('stage')} | attempts={item.get('attempts')}"
    )

    entry = process_mal_id(
        mal_id,
        is_nsfw,
        idx=item.get("index"),
        recovery=True,
    )

    if entry:
        clear_failed_id(mal_id)
        results_by_id[mal_id] = entry
        recovered += 1
        save_outputs(list(results_by_id.values()))

    time.sleep(1.0)

save_outputs(list(results_by_id.values()))

write_section_log(
    "FAILED ID RETRY COMPLETE",
    {
        "Retryable IDs attempted": len(retry_items),
        "Recovered rows": recovered,
        "Remaining failed IDs": len(failed_registry),
        "Output CSV": OUTPUT_CSV,
        "Failed IDs file": FAILED_IDS_FILE,
        "Invalid type IDs file": FILTERED_INVALID_TYPES_FILE,
        "Permanent HTTP skip file": PERMANENT_HTTP_SKIP_FILE,
    }
)

print(f"Recovered rows: {recovered}")
print(f"Remaining failed IDs: {len(failed_registry)}")

Retryable failed IDs: 1


C:\Users\CHAMPUX\AppData\Local\Temp\ipykernel_41684\3889614356.py:26: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_existing = normalize_dataset_columns(pd.read_csv(OUTPUT_CSV))


[2026-05-16 20:23:45.648495] RETRY_SKIPPED_PERMANENT_HTTP_CACHE | MAL 64160
[2026-05-16 20:23:47.399379] ========================================================================
[2026-05-16 20:23:47.400069] FAILED ID RETRY COMPLETE
[2026-05-16 20:23:47.400361] Retryable IDs attempted: 1
[2026-05-16 20:23:47.400655] Recovered rows: 0
[2026-05-16 20:23:47.400879] Remaining failed IDs: 1
[2026-05-16 20:23:47.401089] Output CSV: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\anime_dataset.csv
[2026-05-16 20:23:47.401290] Failed IDs file: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\build\failed_api_requests.json
[2026-05-16 20:23:47.401493] Invalid type IDs file: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\build\skipped_invalid_type_ids.json
[2026-05-16 20:23:47.401708] Permanent HTTP skip file: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\build\skipped_permanent_http_ids.json
[2026-05-16 20:23